# Hodrick–Prescott filter (`hpfilter`) in Python

This notebook ports the provided MATLAB `hpfilter` function to Python, preserving behavior and adding Python-friendly documentation.

## Function behavior

- **Inputs**
  - `y`: original time-series data (`(m,)`, `(m, n)`, or `(n, m)`)
  - `w`: smoothing parameter (common macroeconomics default is `1600` for quarterly data)
  - `plotter`: if `True` or `'makeplot'`, plot each original series against its trend
  - `return_desvabs`: if `True`, also return `desvabs = mean(abs(y-s)/s, axis=0)`

- **Outputs**
  - `s`: filtered (trend) series
  - optionally `desvabs`: standardized absolute difference measure


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import spdiags
from scipy.sparse.linalg import spsolve


def hpfilter(y, w, plotter=False, return_desvabs=False):
    """
    Hodrick-Prescott filter for one or multiple time series.

    Parameters
    ----------
    y : array_like
        Original series. Can be shape (m,), (m, n), or (n, m).
        If rows < cols, the array is transposed to match MATLAB behavior.
    w : float
        Smoothing parameter (e.g., 1600 for quarterly data).
    plotter : bool or str, optional
        If True or 'makeplot', plots original and filtered series.
    return_desvabs : bool, optional
        If True, also returns standardized absolute differences:
        mean(abs(y - s) / s, axis=0).

    Returns
    -------
    s : ndarray
        Filtered trend series with shape (m, n).
    desvabs : ndarray, optional
        Returned only when return_desvabs=True.
    """
    if y is None or w is None:
        raise ValueError('Requires at least two arguments: y and w.')

    y = np.asarray(y, dtype=float)
    if y.ndim == 1:
        y = y[:, None]

    m, n = y.shape
    if m < n:
        y = y.T
        m, n = y.shape

    d = np.tile(np.array([w, -4 * w, (6 * w + 1) / 2.0]), (m, 1))
    d[0, 1] = -2 * w
    d[m - 2, 1] = -2 * w
    d[0, 2] = (1 + w) / 2.0
    d[m - 1, 2] = (1 + w) / 2.0
    d[1, 2] = (5 * w + 1) / 2.0
    d[m - 2, 2] = (5 * w + 1) / 2.0

    # MATLAB: B = spdiags(d, -2:0, m, m); B = B + B'
    B = spdiags(d.T, np.array([-2, -1, 0]), m, m).tocsc()
    B = B + B.T

    s = spsolve(B, y)
    if s.ndim == 1:
        s = s[:, None]

    if plotter is True or plotter == 'makeplot':
        for i in range(n):
            plt.figure(i + 1)
            plt.plot(s[:, i], 'r', label='Filtered trend')
            plt.plot(y[:, i], label='Original series')
            plt.title(f'Series #{i + 1}')
            plt.grid(True)
            plt.legend()

    if return_desvabs:
        desvabs = np.mean(np.abs(y - s) / s, axis=0)
        return s, desvabs

    return s


In [ ]:
# Example usage
np.random.seed(7)
t = np.arange(120)
series = 0.05 * t + np.sin(t / 8) + 0.2 * np.random.randn(len(t))

trend, desvabs = hpfilter(series, 1600, plotter=True, return_desvabs=True)
print('desvabs:', desvabs)
